In [41]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, LSTM, Dropout, Input
from tensorflow.keras.models import Model

In [42]:
df = pd.read_csv('/content/PRSA_data_2010.1.1-2014.12.31.csv')

In [43]:
for col in df.columns:
  print(col, df[col].unique())
  print()

No [    1     2     3 ... 43822 43823 43824]

year [2010 2011 2012 2013 2014]

month [ 1  2  3  4  5  6  7  8  9 10 11 12]

day [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31]

hour [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]

pm2.5 [ nan 129. 148. 159. 181. 138. 109. 105. 124. 120. 132. 140. 152. 164.
 158. 154. 170. 149. 156. 126.  90.  63.  65.  55.  83.  91.  86.  82.
  78.  98. 107.  96.  95.  70.  61.  53.  71.  72.  76.  73.  79.  58.
  25.  26.  28.  20.  29.  27.  32.  30.  31.  33.  34.  36.  39.  41.
  50.  56.  59.  60.  84. 106.  66.  77.  44.  21.  42.  48.  49.  52.
  75.  93. 131. 127. 130.  43.  37.  24.  23.  40.  51.  57.  54.  67.
 198. 190. 210. 195. 275. 110. 100.  81.  92. 135. 155. 250. 200. 231.
 212. 219. 227. 226. 225. 168. 169. 165. 167. 196. 119.  45.  47.  62.
  35.  68.  88.  22.  17.  16.  18.  15.  13.   9.  11.  19.  12. 257.
 174. 161. 137.  64.  87.  89.  94.  69. 102. 141.

In [44]:
df

,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,1,2010,1,1,0,NaN,-21,-11.0,1021.0,NW,1.79,0,0
1,2,2010,1,1,1,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2,3,2010,1,1,2,NaN,-21,-11.0,1019.0,NW,6.71,0,0
3,4,2010,1,1,3,NaN,-21,-14.0,1019.0,NW,9.84,0,0
4,5,2010,1,1,4,NaN,-20,-12.0,1018.0,NW,12.97,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43819,43820,2014,12,31,19,8.0,-23,-2.0,1034.0,NW,231.97,0,0
43820,43821,2014,12,31,20,10.0,-22,-3.0,1034.0,NW,237.78,0,0
43821,43822,2014,12,31,21,10.0,-22,-3.0,1034.0,NW,242.70,0,0
43822,43823,2014,12,31,22,8.0,-22,-4.0,1034.0,NW,246.72,0,0


In [45]:
df.drop(columns=['cbwd', 'No'], inplace = True)

In [46]:
df

,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,Iws,Is,Ir
0,2010,1,1,0,NaN,-21,-11.0,1021.0,1.79,0,0
1,2010,1,1,1,NaN,-21,-12.0,1020.0,4.92,0,0
2,2010,1,1,2,NaN,-21,-11.0,1019.0,6.71,0,0
3,2010,1,1,3,NaN,-21,-14.0,1019.0,9.84,0,0
4,2010,1,1,4,NaN,-20,-12.0,1018.0,12.97,0,0
...,...,...,...,...,...,...,...,...,...,...,...
43819,2014,12,31,19,8.0,-23,-2.0,1034.0,231.97,0,0
43820,2014,12,31,20,10.0,-22,-3.0,1034.0,237.78,0,0
43821,2014,12,31,21,10.0,-22,-3.0,1034.0,242.70,0,0
43822,2014,12,31,22,8.0,-22,-4.0,1034.0,246.72,0,0


In [ ]:
df['pm2.5'] = df['pm2.5'].fillna(df['pm2.5'].mean())
display(df.head())

In [48]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
scaled_df = scalar.fit_transform(df.values)

In [49]:
def create_data(data, seq_len):
  x = []
  y = []
  for i in range(len(data) - seq_len):
    x.append(data[i:i+seq_len, : ])
    y.append(data[i+seq_len, 4])
  return np.array(x), np.array(y)

In [50]:
x, y = create_data(scaled_df, 24)

In [51]:
split_index = int(len(x)*0.8)
xtrain = x[:split_index]
xtest = x[split_index:]
ytrain = y[:split_index]
ytest = y[split_index:]

In [52]:
xtrain.shape

(35040, 24, 11)

In [53]:
D = 11
T = 24

In [54]:
def create_model(T, D):
  i = Input(shape = (T, D))
  x = LSTM(128, return_sequences = False)(i)
  x = Dropout(0.2)(x)
  x = Dense(32, activation='relu')(x)
  x = Dropout(0.2)(x)
  x = Dense(1)(x)

  return Model(i, x)

In [55]:
model = create_model(T, D)

In [56]:
from tensorflow.keras.optimizers import Adam

# Clip gradients so no single update exceeds 1.0
custom_adam = Adam(learning_rate=0.001, clipvalue=1.0)

model.compile(optimizer=custom_adam,
              loss='mse',
              metrics=['mae'])

In [57]:
r = model.fit(xtrain, ytrain, epochs = 100, batch_size = 32, validation_data = (xtest, ytest))

Epoch 1/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - loss: 0.1495 - mae: 0.2440 - val_loss: 0.0672 - val_mae: 0.1591
Epoch 2/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.1057 - mae: 0.1961 - val_loss: 0.0689 - val_mae: 0.1543
Epoch 3/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.1014 - mae: 0.1890 - val_loss: 0.0645 - val_mae: 0.1544
Epoch 4/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0985 - mae: 0.1866 - val_loss: 0.0715 - val_mae: 0.1627
Epoch 5/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.0971 - mae: 0.1853 - val_loss: 0.0737 - val_mae: 0.1669
Epoch 6/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0980 - mae: 0.1848 - val_loss: 0.0625 - val_mae: 0.1416
Epoch 7/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0948 - mae: 0.1831 - val_loss: 0.0655 - val_mae: 0.1494
Epoch 8/100
1095/1095 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0941 - mae: 0.1825 - val_loss: 0.0664 - val_mae: 0.1471
Epoch 9/100
1095/1095 ━━━━━━━━━━

KeyboardInterrupt: 